# Object Detection

In [10]:
# Imports

import cv2
import numpy
import sys

In [11]:
# Global variables

modelFile = "../../Models/ssd_mobilenet_v2_coco_2018_03_29/frozen_inference_graph.pb"
configFile = "../../Models/ssd_mobilenet_v2_coco_2018_03_29.pbtxt"

classFile = "coco_class_labels.txt"
with open(classFile, "rt") as f:
    labels = f.read().split("\n")
print("Labels:", labels)

dimensions = (300, 300)

confThreshold=0.1

fontFace = cv2.FONT_HERSHEY_SIMPLEX
fontScale = 0.7
thickness = 1

Labels: ['unlabeled', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'street sign', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'hat', 'backpack', 'umbrella', 'shoe', 'eye glasses', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'plate', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'mirror', 'dining table', 'window', 'desk', 'toilet', 'door', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'blender', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush', 'hair brush', ''

In [12]:
# Initialize model

net = cv2.dnn.readNetFromTensorflow(modelFile, configFile)

In [13]:
# Functions

def detectObject(net, frame):

    blob = cv2.dnn.blobFromImage(frame, 1.0, size=dimensions, mean=(0,0,0), swapRB=True, crop=False)

    net.setInput(blob)

    objects = net.forward()

    return objects

def displayText(frame, text, x, y):

    textSize = cv2.getTextSize(text, fontFace, fontScale, thickness)
    dim = textSize[0]
    baseline = textSize[1]

    cv2.rectangle(frame, (x, y - dim[1] - baseline), (x + dim[0], y + baseline), (0, 0, 0), cv2.FILLED)
    cv2.putText(frame, text, (x, y-5), fontFace, fontScale, (0, 255, 255), thickness, cv2.LINE_AA)

def displayObjects(frame, objects):
    rows, cols = frame.shape[:2]

    for i in range(objects.shape[2]):
        confScore = float(objects[0, 0, i, 2])

        if confScore > confThreshold:
            classId = int(objects[0, 0, i, 1])
            xLeftBottom = int(objects[0, 0, i, 3] * cols)
            yLeftBottom = int(objects[0, 0, i, 4] * rows)
            xRightTop   = int(objects[0, 0, i, 5] * cols)
            yRightTop   = int(objects[0, 0, i, 6] * rows)

            displayText(frame, labels[classId] + ", Confidence:" + str(confScore), xLeftBottom, yLeftBottom - 10)
            cv2.rectangle(frame, (xLeftBottom, yLeftBottom), (xRightTop, yRightTop), (255, 255, 255), 2)
            cv2.rectangle(frame, (xLeftBottom, yLeftBottom), (xRightTop, yRightTop), (0, 255, 0))

In [14]:
source = cv2.VideoCapture(1)

window_name = "Object Detection"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

while cv2.waitKey(1) != 27:
    read_success, frame = source.read()
    if not read_success:
        break

    #frame = cv2.flip(frame, 1)

    objects = detectObject(net, frame)
    displayObjects(frame, objects)

    cv2.imshow(window_name, frame)